In [1]:
# Connect the flip mount
from pylablib.devices.Thorlabs import MFF

serial = "37000705"
flip_mount = MFF(serial)
flip_mount.open()  # explicitly open the FTDI backend




C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\pylablib\devices\Thorlabs\kinesis.py:230: UserWarning: model number MFF002 doesn't match the device ID prefix 37(MFF10.)
  warnings.warn("model number {} doesn't match the device ID prefix {}({})".format(model_no,port,port_model_no))


In [2]:
# Move the flip, 0: block, 1:remove
flip_mount.move_to_state(0) #Block


In [3]:
flip_mount.move_to_state(1) #Remove


In [4]:
# Connect the power meter
from pylablib.devices.Thorlabs import PM160

# Use the serial number exactly as shown on the device
#serial = "P0005244"
pm = PM160('USB0::0x1313::0x8078::P0005244::0::INSTR')

# Open the connection
pm.open()
pm.enable_autorange

<bound method GenericPM.enable_autorange of <pylablib.devices.Thorlabs.misc.PM160 object at 0x000001C51F206B90>>

In [5]:
# Check the settings
pm.get_full_status(include="all")


{'mode': 'power',
 'autorange': True,
 'wavelength': 6.33e-07,
 'range': 0.0001477929,
 'reading': 3.04657569e-05}

In [7]:
# Change wavelength
pm.set_wavelength(5.32e-07)

5.32e-07

In [8]:
# Read power (in Watts)
power = pm.get_power()
print(f"Measured power: {power} W")

Measured power: 6.47766574e-05 W


In [9]:
import pyvisa as pvisa
rm = pvisa.ResourceManager()
resources = rm.list_resources()
print("List")
for res in resources:
    print(res)

List
GPIB0::13::INSTR
GPIB0::2::INSTR
GPIB1::3::INSTR
GPIB1::8::INSTR
PXI0::4-0.0::INSTR
USB0::0x05E6::0x2636::4353546::0::INSTR
USB0::0x05E6::0x2636::4437673::0::INSTR
USB0::0x0957::0x3D18::MY59030016::0::INSTR
USB0::0x1313::0x8078::P0005244::0::INSTR
USB0::0x1313::0x8078::PM001388::0::INSTR
USB0::0x2EC7::0x6522::802557043747210033::0::INSTR
ASRL3::INSTR
ASRL5::INSTR


In [41]:
import pyvisa as pvisa
rm = pvisa.ResourceManager()
resources = rm.list_resources()
print("List")
for res in resources:
    print(res)

List
GPIB0::13::INSTR
GPIB0::2::INSTR
GPIB1::3::INSTR
GPIB1::8::INSTR
USB0::0x05E6::0x2636::4353546::0::INSTR
USB0::0x05E6::0x2636::4437673::0::INSTR
USB0::0x0957::0x3D18::MY59030016::0::INSTR
USB0::0x1313::0x8078::P0005244::0::INSTR
USB0::0x1313::0x8078::PM001388::0::INSTR
USB0::0x2EC7::0x6522::802557043747210033::0::INSTR
ASRL3::INSTR
ASRL5::INSTR


In [8]:
help(power_meter)

Help on PM160 in module pylablib.devices.Thorlabs.misc object:

class PM160(GenericPM)
 |  PM160(addr)
 |  
 |  Thorlabs PM160 optical Power Meter.
 |  
 |  Args:
 |      addr: connection address (usually, a VISA connection string or a COM port for bluetooth devices)
 |  
 |  Method resolution order:
 |      PM160
 |      GenericPM
 |      pylablib.core.devio.SCPI.SCPIDevice
 |      pylablib.core.devio.comm_backend.ICommBackendWrapper
 |      pylablib.core.devio.interface.IDevice
 |      builtins.object
 |  
 |  Methods inherited from GenericPM:
 |  
 |  __init__(self, addr)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  enable_autorange(self, enable=True, sensor_mode=None)
 |      Enable or disable autorange for the given sensor mode.
 |      
 |      If `sensor_mode` is ``None``, set value for the current sensor mode.
 |  
 |  get_device_info(self)
 |      Get device info.
 |      
 |      Return tuple ``(manufacturer, name, serial, firmware)``.
 |  


In [5]:
import nidaqmx
import time

In [36]:
# Device name (check NI MAX, usually "Dev1")
DEV = "Dev3"

# We want lines 0, 1, 2 on port2
LINES = f"{DEV}/port2/line0:2"  # shorthand for line0 through line2

with nidaqmx.Task() as task:
    task.do_channels.add_do_chan(LINES)

    # ---- Initial state ----
    task.write([False, False, False], auto_start=True)
    # PUL=False, ENA=False (enabled), DIR=False

    print("Motor enabled, direction set.")

    steps = 20000
    delay = 0.0

    for i in range(steps):
        # Pulse HIGH
        task.write([False, False, True], auto_start=True)  
        time.sleep(delay)
        # Pulse LOW
        task.write([False, False, False], auto_start=True)
        time.sleep(delay)

    print("Done stepping.")

    # Disable motor
    task.write([False, True, False], auto_start=True)  # ENA=HIGH (disable)
    print("Motor disabled.")

Motor enabled, direction set.
Done stepping.
Motor disabled.


In [16]:
import nidaqmx, time

DEV = "Dev3"
LINES = f"{DEV}/port2/line0:2"

with nidaqmx.Task() as task:
    task.do_channels.add_do_chan(LINES)

    # Enable driver, DIR=0
    task.write([False, False, False], auto_start=True)
    time.sleep(0.5)

    print("Sending 10 pulses...")
    for i in range(10):
        task.write([True, False, False], auto_start=True)  
        time.sleep(0.1)  # 100 ms HIGH
        task.write([False, False, False], auto_start=True)
        time.sleep(0.1)  # 100 ms LOW


Sending 10 pulses...


In [30]:
import nidaqmx

DEV = "Dev3"
LINES = f"{DEV}/port2/line0:2"

with nidaqmx.Task() as task:
    task.do_channels.add_do_chan(LINES)

    # Example: PUL=LOW, ENA=LOW (enabled), DIR=LOW
    #task.write([False, False, False], auto_start=True)
    #task.write([False, True, False], auto_start=True)   # ENA=HIGH
    #task.write([False, False, True], auto_start=True)   # DIR=HIGH
    task.write([True, False, False], auto_start=True)   # PUL=HIGH

    print("Set: PUL=0, ENA=0, DIR=0")

    input("Press Enter to exit (lines will reset)...")


Set: PUL=0, ENA=0, DIR=0


KeyboardInterrupt: Interrupted by user

In [ ]:
import nidaqmx
from nidaqmx.constants import LineGrouping
import time

def read_port_lines(dev='Dev1', port=0):
    with nidaqmx.Task() as task:
        # add each line individually to force boolean list reading
        task.di_channels.add_di_chan(f"{dev}/port{port}/line0:7", line_grouping=LineGrouping.CHAN_PER_LINE)
        vals = task.read()  # returns list of booleans
    return vals

def pretty_print_all_ports(dev='Dev1'):
    for p in range(3):  # USB-6501 has port0, port1, port2
        vals = read_port_lines(dev=dev, port=p)
        highs = [i for i, v in enumerate(vals) if v]
        print(f"Port{p} HIGH lines: {highs}  (raw: {vals})")

if __name__ == "__main__":
    pretty_print_all_ports()


In [21]:
import nidaqmx
import time
with nidaqmx.Task() as task:
    task.do_channels.add_do_chan("Dev3/port2/line7")
    for i in range(1000):
        task.write(True)
        time.sleep(0.005)
        task.write(False)
        time.sleep(0.005)
        

In [25]:
# port2_map_test.py
import nidaqmx
import time
import csv
from nidaqmx.constants import LineGrouping

DEVICE = "Dev3"
PORT = 2                # you're using Port2
LINE4_INDEX = 4         # Port2/line4 is the controller status input
OVERRIDE_LINE = 7       # Port2/line7 looks like ENABLE/RESET (keep LOW to allow)
CANDIDATES = [0,1,2,3,5,6]  # test these (exclude 4, and you can exclude 7 as it's override)
PULSES_PER_TEST = 200   # number of pulses per candidate (short bursts)
PULSE_FREQ_HZ = 1000     # pulse frequency (Hz) -> period = 1/F
DUTY = 0.5              # duty cycle (fraction)
INTER_TEST_PAUSE = 1.0  # seconds pause between testing different lines
CSV_FILE = "port2_line_mapping_log.csv"

period = 1.0 / PULSE_FREQ_HZ
high_time = period * DUTY
low_time = period * (1.0 - DUTY)

def safe_write_bool(task, val):
    # writing a single boolean to a single line channel
    task.write(bool(val))

def pulse_and_log(candidate_line):
    chan_out = f"{DEVICE}/port{PORT}/line{candidate_line}"
    chan_override = f"{DEVICE}/port{PORT}/line{OVERRIDE_LINE}"
    chan_in = f"{DEVICE}/port{PORT}/line{LINE4_INDEX}"

    # We will create three tasks: one input (read line4), one for override line (set low),
    # and one for candidate output pulses.
    with nidaqmx.Task() as t_in, nidaqmx.Task() as t_override, nidaqmx.Task() as t_out:
        t_in.di_channels.add_di_chan(chan_in, line_grouping=LineGrouping.CHAN_PER_LINE)
        t_override.do_channels.add_do_chan(chan_override, line_grouping=LineGrouping.CHAN_PER_LINE)
        t_out.do_channels.add_do_chan(chan_out, line_grouping=LineGrouping.CHAN_PER_LINE)

        # Ensure override = LOW (allow controller)
        safe_write_bool(t_override, False)
        time.sleep(0.05)

        # Small settle
        before = time.time()
        # Start pulsing
        print(f"\nTesting candidate: {chan_out} -> pulsing {PULSES_PER_TEST} pulses @ {PULSE_FREQ_HZ} Hz")
        for i in range(PULSES_PER_TEST):
            t_out.write(True)
            t_in_state = t_in.read()
            tstamp = time.time()
            log_row = [tstamp, f"port{PORT}/line{candidate_line}", "HIGH", i, t_in_state]
            writer.writerow(log_row)
            print(f"{i:03d} OUT={1}  | feedback line4={t_in_state}  @ {tstamp:.4f}")
            time.sleep(high_time)

            t_out.write(False)
            t_in_state = t_in.read()
            tstamp = time.time()
            log_row = [tstamp, f"port{PORT}/line{candidate_line}", "LOW", i, t_in_state]
            writer.writerow(log_row)
            print(f"{i:03d} OUT={0}  | feedback line4={t_in_state}  @ {tstamp:.4f}")
            time.sleep(low_time)

        # leave override LOW, then short settle
        time.sleep(0.05)

if __name__ == "__main__":
    print("=== Port2 line mapping test ===")
    print("Candidates to test (port2):", CANDIDATES)
    print("Make sure GNDs are common and you have series resistor in place for outputs.")
    print("Logging to:", CSV_FILE)
    with open(CSV_FILE, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["timestamp", "candidate_line", "out_state", "pulse_index", "line4_feedback"])
        # Bind writer into global for function access
        globals()["writer"] = writer

        for line in CANDIDATES:
            # Double check override forced LOW before each test
            # (this will be done in pulse_and_log)
            pulse_and_log(line)
            print(f"Finished testing line {line}. Pausing {INTER_TEST_PAUSE}s before next one.")
            time.sleep(INTER_TEST_PAUSE)

    print("All done. Inspect CSV for line4 responses and timestamps.")


=== Port2 line mapping test ===
Candidates to test (port2): [0, 1, 2, 3, 5, 6]
Make sure GNDs are common and you have series resistor in place for outputs.
Logging to: port2_line_mapping_log.csv

Testing candidate: Dev3/port2/line0 -> pulsing 200 pulses @ 1000 Hz
000 OUT=1  | feedback line4=True  @ 1756799767.6066
000 OUT=0  | feedback line4=True  @ 1756799767.6226
001 OUT=1  | feedback line4=True  @ 1756799767.6391
001 OUT=0  | feedback line4=True  @ 1756799767.6547
002 OUT=1  | feedback line4=True  @ 1756799767.6684
002 OUT=0  | feedback line4=True  @ 1756799767.6842
003 OUT=1  | feedback line4=True  @ 1756799767.6992
003 OUT=0  | feedback line4=True  @ 1756799767.7150
004 OUT=1  | feedback line4=True  @ 1756799767.7328
004 OUT=0  | feedback line4=True  @ 1756799767.7464
005 OUT=1  | feedback line4=True  @ 1756799767.7629
005 OUT=0  | feedback line4=True  @ 1756799767.7781
006 OUT=1  | feedback line4=True  @ 1756799767.7934
006 OUT=0  | feedback line4=True  @ 1756799767.8086
007 OUT=

078 OUT=1  | feedback line4=True  @ 1756799770.0533
078 OUT=0  | feedback line4=True  @ 1756799770.0693
079 OUT=1  | feedback line4=True  @ 1756799770.0851
079 OUT=0  | feedback line4=True  @ 1756799770.1010
080 OUT=1  | feedback line4=True  @ 1756799770.1170
080 OUT=0  | feedback line4=True  @ 1756799770.1325
081 OUT=1  | feedback line4=True  @ 1756799770.1484
081 OUT=0  | feedback line4=True  @ 1756799770.1644
082 OUT=1  | feedback line4=True  @ 1756799770.1793
082 OUT=0  | feedback line4=True  @ 1756799770.1963
083 OUT=1  | feedback line4=True  @ 1756799770.2123
083 OUT=0  | feedback line4=True  @ 1756799770.2270
084 OUT=1  | feedback line4=True  @ 1756799770.2440
084 OUT=0  | feedback line4=True  @ 1756799770.2599
085 OUT=1  | feedback line4=True  @ 1756799770.2759
085 OUT=0  | feedback line4=True  @ 1756799770.2917
086 OUT=1  | feedback line4=True  @ 1756799770.3078
086 OUT=0  | feedback line4=True  @ 1756799770.3230
087 OUT=1  | feedback line4=True  @ 1756799770.3392
087 OUT=0  |

160 OUT=1  | feedback line4=True  @ 1756799772.6206
160 OUT=0  | feedback line4=True  @ 1756799772.6356
161 OUT=1  | feedback line4=True  @ 1756799772.6511
161 OUT=0  | feedback line4=True  @ 1756799772.6666
162 OUT=1  | feedback line4=True  @ 1756799772.6824
162 OUT=0  | feedback line4=True  @ 1756799772.6983
163 OUT=1  | feedback line4=True  @ 1756799772.7144
163 OUT=0  | feedback line4=True  @ 1756799772.7323
164 OUT=1  | feedback line4=True  @ 1756799772.7475
164 OUT=0  | feedback line4=True  @ 1756799772.7631
165 OUT=1  | feedback line4=True  @ 1756799772.7791
165 OUT=0  | feedback line4=True  @ 1756799772.7941
166 OUT=1  | feedback line4=True  @ 1756799772.8106
166 OUT=0  | feedback line4=True  @ 1756799772.8274
167 OUT=1  | feedback line4=True  @ 1756799772.8420
167 OUT=0  | feedback line4=True  @ 1756799772.8566
168 OUT=1  | feedback line4=True  @ 1756799772.8727
168 OUT=0  | feedback line4=True  @ 1756799772.8868
169 OUT=1  | feedback line4=True  @ 1756799772.9040
169 OUT=0  |

041 OUT=1  | feedback line4=True  @ 1756799776.2976
041 OUT=0  | feedback line4=True  @ 1756799776.3116
042 OUT=1  | feedback line4=True  @ 1756799776.3253
042 OUT=0  | feedback line4=True  @ 1756799776.3425
043 OUT=1  | feedback line4=True  @ 1756799776.3580
043 OUT=0  | feedback line4=True  @ 1756799776.3734
044 OUT=1  | feedback line4=True  @ 1756799776.3891
044 OUT=0  | feedback line4=True  @ 1756799776.4063
045 OUT=1  | feedback line4=True  @ 1756799776.4204
045 OUT=0  | feedback line4=True  @ 1756799776.4354
046 OUT=1  | feedback line4=True  @ 1756799776.4516
046 OUT=0  | feedback line4=True  @ 1756799776.4663
047 OUT=1  | feedback line4=True  @ 1756799776.4818
047 OUT=0  | feedback line4=True  @ 1756799776.4974
048 OUT=1  | feedback line4=True  @ 1756799776.5138
048 OUT=0  | feedback line4=True  @ 1756799776.5268
049 OUT=1  | feedback line4=True  @ 1756799776.5426
049 OUT=0  | feedback line4=True  @ 1756799776.5579
050 OUT=1  | feedback line4=True  @ 1756799776.5732
050 OUT=0  |

123 OUT=0  | feedback line4=True  @ 1756799778.8681
124 OUT=1  | feedback line4=True  @ 1756799778.8831
124 OUT=0  | feedback line4=True  @ 1756799778.8982
125 OUT=1  | feedback line4=True  @ 1756799778.9145
125 OUT=0  | feedback line4=True  @ 1756799778.9296
126 OUT=1  | feedback line4=True  @ 1756799778.9448
126 OUT=0  | feedback line4=True  @ 1756799778.9612
127 OUT=1  | feedback line4=True  @ 1756799778.9765
127 OUT=0  | feedback line4=True  @ 1756799778.9922
128 OUT=1  | feedback line4=True  @ 1756799779.0088
128 OUT=0  | feedback line4=True  @ 1756799779.0242
129 OUT=1  | feedback line4=True  @ 1756799779.0392
129 OUT=0  | feedback line4=True  @ 1756799779.0560
130 OUT=1  | feedback line4=True  @ 1756799779.0711
130 OUT=0  | feedback line4=True  @ 1756799779.0859
131 OUT=1  | feedback line4=True  @ 1756799779.1019
131 OUT=0  | feedback line4=True  @ 1756799779.1183
132 OUT=1  | feedback line4=True  @ 1756799779.1351
132 OUT=0  | feedback line4=True  @ 1756799779.1507
133 OUT=1  |

...

013 OUT=0  | feedback line4=False  @ 1756799797.6203
014 OUT=1  | feedback line4=False  @ 1756799797.6356
014 OUT=0  | feedback line4=False  @ 1756799797.6520
015 OUT=1  | feedback line4=False  @ 1756799797.6678
015 OUT=0  | feedback line4=False  @ 1756799797.6841
016 OUT=1  | feedback line4=False  @ 1756799797.6986
016 OUT=0  | feedback line4=False  @ 1756799797.7149
017 OUT=1  | feedback line4=False  @ 1756799797.7309
017 OUT=0  | feedback line4=False  @ 1756799797.7474
018 OUT=1  | feedback line4=False  @ 1756799797.7620
018 OUT=0  | feedback line4=False  @ 1756799797.7778
019 OUT=1  | feedback line4=False  @ 1756799797.7943
019 OUT=0  | feedback line4=False  @ 1756799797.8097
020 OUT=1  | feedback line4=False  @ 1756799797.8254
020 OUT=0  | feedback line4=False  @ 1756799797.8416
021 OUT=1  | feedback line4=False  @ 1756799797.8570
021 OUT=0  | feedback line4=False  @ 1756799797.8729
022 OUT=1  | feedback line4=False  @ 1756799797.8890
022 OUT=0  | feedback line4=False  @ 175679979

096 OUT=0  | feedback line4=False  @ 1756799800.2124
097 OUT=1  | feedback line4=False  @ 1756799800.2307
097 OUT=0  | feedback line4=False  @ 1756799800.2424
098 OUT=1  | feedback line4=False  @ 1756799800.2585
098 OUT=0  | feedback line4=False  @ 1756799800.2739
099 OUT=1  | feedback line4=False  @ 1756799800.2887
099 OUT=0  | feedback line4=False  @ 1756799800.3036
100 OUT=1  | feedback line4=False  @ 1756799800.3221
100 OUT=0  | feedback line4=False  @ 1756799800.3383
101 OUT=1  | feedback line4=False  @ 1756799800.3534
101 OUT=0  | feedback line4=False  @ 1756799800.3689
102 OUT=1  | feedback line4=False  @ 1756799800.3843
102 OUT=0  | feedback line4=False  @ 1756799800.4000
103 OUT=1  | feedback line4=False  @ 1756799800.4155
103 OUT=0  | feedback line4=False  @ 1756799800.4304
104 OUT=1  | feedback line4=False  @ 1756799800.4447
104 OUT=0  | feedback line4=False  @ 1756799800.4601
105 OUT=1  | feedback line4=False  @ 1756799800.4761
105 OUT=0  | feedback line4=False  @ 175679980

177 OUT=1  | feedback line4=False  @ 1756799802.7338
177 OUT=0  | feedback line4=False  @ 1756799802.7508
178 OUT=1  | feedback line4=False  @ 1756799802.7666
178 OUT=0  | feedback line4=False  @ 1756799802.7831
179 OUT=1  | feedback line4=False  @ 1756799802.7989
179 OUT=0  | feedback line4=False  @ 1756799802.8134
180 OUT=1  | feedback line4=False  @ 1756799802.8290
180 OUT=0  | feedback line4=False  @ 1756799802.8438
181 OUT=1  | feedback line4=False  @ 1756799802.8593
181 OUT=0  | feedback line4=False  @ 1756799802.8770
182 OUT=1  | feedback line4=False  @ 1756799802.8910
182 OUT=0  | feedback line4=False  @ 1756799802.9074
183 OUT=1  | feedback line4=False  @ 1756799802.9212
183 OUT=0  | feedback line4=False  @ 1756799802.9376
184 OUT=1  | feedback line4=False  @ 1756799802.9529
184 OUT=0  | feedback line4=False  @ 1756799802.9687
185 OUT=1  | feedback line4=False  @ 1756799802.9851
185 OUT=0  | feedback line4=False  @ 1756799803.0002
186 OUT=1  | feedback line4=False  @ 175679980

054 OUT=1  | feedback line4=False  @ 1756799806.2753
054 OUT=0  | feedback line4=False  @ 1756799806.2903
055 OUT=1  | feedback line4=False  @ 1756799806.3061
055 OUT=0  | feedback line4=False  @ 1756799806.3239
056 OUT=1  | feedback line4=False  @ 1756799806.3389
056 OUT=0  | feedback line4=False  @ 1756799806.3555
057 OUT=1  | feedback line4=False  @ 1756799806.3712
057 OUT=0  | feedback line4=False  @ 1756799806.3864
058 OUT=1  | feedback line4=False  @ 1756799806.4023
058 OUT=0  | feedback line4=False  @ 1756799806.4174
059 OUT=1  | feedback line4=False  @ 1756799806.4327
059 OUT=0  | feedback line4=False  @ 1756799806.4475
060 OUT=1  | feedback line4=False  @ 1756799806.4644
060 OUT=0  | feedback line4=False  @ 1756799806.4782
061 OUT=1  | feedback line4=False  @ 1756799806.4945
061 OUT=0  | feedback line4=False  @ 1756799806.5099
062 OUT=1  | feedback line4=False  @ 1756799806.5259
062 OUT=0  | feedback line4=False  @ 1756799806.5410
063 OUT=1  | feedback line4=False  @ 175679980

136 OUT=1  | feedback line4=False  @ 1756799808.8353
136 OUT=0  | feedback line4=False  @ 1756799808.8497
137 OUT=1  | feedback line4=False  @ 1756799808.8648
137 OUT=0  | feedback line4=False  @ 1756799808.8814
138 OUT=1  | feedback line4=False  @ 1756799808.8961
138 OUT=0  | feedback line4=False  @ 1756799808.9125
139 OUT=1  | feedback line4=False  @ 1756799808.9300
139 OUT=0  | feedback line4=False  @ 1756799808.9469
140 OUT=1  | feedback line4=False  @ 1756799808.9621
140 OUT=0  | feedback line4=False  @ 1756799808.9775
141 OUT=1  | feedback line4=False  @ 1756799808.9932
141 OUT=0  | feedback line4=False  @ 1756799809.0082
142 OUT=1  | feedback line4=False  @ 1756799809.0232
142 OUT=0  | feedback line4=False  @ 1756799809.0373
143 OUT=1  | feedback line4=False  @ 1756799809.0531
143 OUT=0  | feedback line4=False  @ 1756799809.0690
144 OUT=1  | feedback line4=False  @ 1756799809.0856
144 OUT=0  | feedback line4=False  @ 1756799809.0998
145 OUT=1  | feedback line4=False  @ 175679980